In [ ]:
import pandas as pd
import numpy as np

train_ratings = pd.read_csv(
    "../datasets/processed/train_ratings.csv"
)

test_ratings = pd.read_csv(
    "../datasets/processed/test_ratings.csv"
)

print("Train shape:", train_ratings.shape)
print("Test shape:", test_ratings.shape)

In [ ]:
# Jedinstveni korisnici i recepti
user_ids = train_ratings["user_id"].unique()
recipe_ids = train_ratings["recipe_id"].unique()

num_users = len(user_ids)
num_recipes = len(recipe_ids)

print("Number of users:", num_users)
print("Number of recipes:", num_recipes)

In [ ]:
# Mapiranje originalnih ID-jeva na indekse za neuronsku mrežu

user_to_index = {
    user_id: index
    for index, user_id in enumerate(user_ids)
}

recipe_to_index = {
    recipe_id: index
    for index, recipe_id in enumerate(recipe_ids)
}

print("User mapping:", list(user_to_index.items())[:5])
print("Recipe mapping:", list(recipe_to_index.items())[:5])

In [ ]:
# Kopija trening podataka
ncf_train = train_ratings[["user_id", "recipe_id", "rating"]].copy()

# Originalne ID-jeve pretvaramo u indekse
ncf_train["user_index"] = ncf_train["user_id"].map(user_to_index)
ncf_train["recipe_index"] = ncf_train["recipe_id"].map(recipe_to_index)

print(ncf_train.head())
print("Training samples:", len(ncf_train))

In [ ]:
print("Missing user indexes:", ncf_train["user_index"].isna().sum())
print("Missing recipe indexes:", ncf_train["recipe_index"].isna().sum())

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader


class RecipeRatingDataset(Dataset):

    def __init__(self, dataframe):
        self.users = torch.tensor(
            dataframe["user_index"].values,
            dtype=torch.long
        )

        self.recipes = torch.tensor(
            dataframe["recipe_index"].values,
            dtype=torch.long
        )

        self.ratings = torch.tensor(
            dataframe["rating"].values,
            dtype=torch.float32
        )

    def __len__(self):
        return len(self.ratings)

    def __getitem__(self, index):
        return (
            self.users[index],
            self.recipes[index],
            self.ratings[index]
        )

In [ ]:
train_dataset = RecipeRatingDataset(ncf_train)

train_loader = DataLoader(
    train_dataset,
    batch_size=256,
    shuffle=True
)

print("Number of training samples:", len(train_dataset))

In [ ]:
users, recipes_batch, ratings = next(iter(train_loader))

print("Users shape:", users.shape)
print("Recipes shape:", recipes_batch.shape)
print("Ratings shape:", ratings.shape)

print("\nFirst 5 users:", users[:5])
print("First 5 recipes:", recipes_batch[:5])
print("First 5 ratings:", ratings[:5])

In [ ]:
import torch.nn as nn


class NCF(nn.Module):

    def __init__(
            self,
            num_users,
            num_recipes,
            embedding_dim=32
    ):
        super().__init__()

        # User embedding
        self.user_embedding = nn.Embedding(
            num_users,
            embedding_dim
        )

        # Recipe embedding
        self.recipe_embedding = nn.Embedding(
            num_recipes,
            embedding_dim
        )

        # Neural network
        self.mlp = nn.Sequential(
            nn.Linear(embedding_dim * 2, 64),
            nn.ReLU(),

            nn.Linear(64, 32),
            nn.ReLU(),

            nn.Linear(32, 16),
            nn.ReLU(),

            nn.Linear(16, 1)
        )

    def forward(self, user, recipe):

        user_vector = self.user_embedding(user)
        recipe_vector = self.recipe_embedding(recipe)

        # Spojimo user i recipe embedding
        x = torch.cat(
            [user_vector, recipe_vector],
            dim=1
        )

        # Neural network
        output = self.mlp(x)

        return output.squeeze(1)

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)

model = NCF(
    num_users=num_users,
    num_recipes=num_recipes,
    embedding_dim=32
).to(device)

print(model)

In [ ]:
criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

print("Model parameters:", sum(
    p.numel()
    for p in model.parameters()
))

In [ ]:
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU nije dostupan za PyTorch.")

In [ ]:
from torch.utils.data import Dataset, DataLoader


class RecipeRatingDataset(Dataset):

    def __init__(self, ratings):
        self.users = torch.tensor(
            ratings["user_idx"].values,
            dtype=torch.long
        )

        self.recipes = torch.tensor(
            ratings["recipe_idx"].values,
            dtype=torch.long
        )

        self.ratings = torch.tensor(
            ratings["rating"].values,
            dtype=torch.float32
        )

    def __len__(self):
        return len(self.ratings)

    def __getitem__(self, idx):
        return (
            self.users[idx],
            self.recipes[idx],
            self.ratings[idx]
        )

In [ ]:
print(train_ratings.columns.tolist())

In [ ]:
# Mapiranje stvarnih ID-jeva na indekse koje PyTorch Embedding koristi

user_to_idx = {
    user_id: idx
    for idx, user_id in enumerate(train_ratings["user_id"].unique())
}

recipe_to_idx = {
    recipe_id: idx
    for idx, recipe_id in enumerate(train_ratings["recipe_id"].unique())
}

# Dodaj indekse u training skup
train_ratings["user_idx"] = train_ratings["user_id"].map(user_to_idx)
train_ratings["recipe_idx"] = train_ratings["recipe_id"].map(recipe_to_idx)

print("Number of users:", len(user_to_idx))
print("Number of recipes:", len(recipe_to_idx))

print(train_ratings[
          ["user_id", "user_idx", "recipe_id", "recipe_idx", "rating"]
      ].head())

In [ ]:

print("Da li je GPU dostupan:", torch.cuda.is_available())
     

In [ ]:
from torch.utils.data import Dataset, DataLoader


class RecipeRatingDataset(Dataset):

    def __init__(self, ratings):
        self.users = torch.tensor(
            ratings["user_idx"].values,
            dtype=torch.long
        )

        self.recipes = torch.tensor(
            ratings["recipe_idx"].values,
            dtype=torch.long
        )

        self.ratings = torch.tensor(
            ratings["rating"].values,
            dtype=torch.float32
        )

    def __len__(self):
        return len(self.ratings)

    def __getitem__(self, idx):
        return (
            self.users[idx],
            self.recipes[idx],
            self.ratings[idx]
        )


train_dataset = RecipeRatingDataset(train_ratings)

train_loader = DataLoader(
    train_dataset,
    batch_size=1024,
    shuffle=True
)

print("Dataset size:", len(train_dataset))
print("Number of batches:", len(train_loader))

In [ ]:
users, recipes_batch, ratings_batch = next(iter(train_loader))

print("Users shape:", users.shape)
print("Recipes shape:", recipes_batch.shape)
print("Ratings shape:", ratings_batch.shape)

print("\nFirst 5 users:", users[:5])
print("First 5 recipes:", recipes_batch[:5])
print("First 5 ratings:", ratings_batch[:5])

In [ ]:
model.eval()

with torch.no_grad():

    predictions = model(
        users.to(device),
        recipes_batch.to(device)
    )

print("Predictions shape:", predictions.shape)
print("First 10 predictions:", predictions[:10])

In [ ]:
import time

num_epochs = 5

model.train()

for epoch in range(num_epochs):

    start_time = time.time()

    total_loss = 0.0

    for users, recipes_batch, ratings_batch in train_loader:

        users = users.to(device)
        recipes_batch = recipes_batch.to(device)
        ratings_batch = ratings_batch.to(device)

        # Predikcija
        predictions = model(
            users,
            recipes_batch
        )

        # Greška
        loss = criterion(
            predictions,
            ratings_batch
        )

        # Reset gradienta
        optimizer.zero_grad()

        # Backpropagation
        loss.backward()

        # Update parametara
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    elapsed = time.time() - start_time

    print(
        f"Epoch {epoch + 1}/{num_epochs} "
        f"- Loss: {avg_loss:.4f} "
        f"- Time: {elapsed:.1f}s"
    )

In [ ]:
import torch
import sys

print("Python:", sys.executable)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
test_ratings["user_idx"] = test_ratings["user_id"].map(user_to_idx)
test_ratings["recipe_idx"] = test_ratings["recipe_id"].map(recipe_to_idx)

print("Test shape:", test_ratings.shape)
print(
    "Users missing:",
    test_ratings["user_idx"].isna().sum()
)
print(
    "Recipes missing:",
    test_ratings["recipe_idx"].isna().sum()
)

In [ ]:
test_dataset = RecipeRatingDataset(test_ratings)

test_loader = DataLoader(
    test_dataset,
    batch_size=1024,
    shuffle=False
)

print("Test samples:", len(test_dataset))
print("Test batches:", len(test_loader))

In [ ]:
model.eval()

total_test_loss = 0.0

with torch.no_grad():

    for users, recipes_batch, ratings_batch in test_loader:

        users = users.to(device)
        recipes_batch = recipes_batch.to(device)
        ratings_batch = ratings_batch.to(device)

        predictions = model(
            users,
            recipes_batch
        )

        loss = criterion(
            predictions,
            ratings_batch
        )

        total_test_loss += loss.item()

avg_test_loss = total_test_loss / len(test_loader)

print(f"Test Loss: {avg_test_loss:.4f}")

In [ ]:
# Recepti koje je svaki korisnik već ocenio u trening skupu
user_seen_recipes = (
    train_ratings
    .groupby("user_idx")["recipe_idx"]
    .apply(set)
    .to_dict()
)

print("Users with training history:", len(user_seen_recipes))

In [ ]:
def recommend_ncf(user_idx, model, num_recipes, user_seen_recipes, top_k=10):

    model.eval()

    seen = user_seen_recipes.get(user_idx, set())

    # Kandidati su recepti koje korisnik još nije ocenio
    candidate_recipes = [
        recipe_idx
        for recipe_idx in range(num_recipes)
        if recipe_idx not in seen
    ]

    users = torch.tensor(
        [user_idx] * len(candidate_recipes),
        dtype=torch.long
    ).to(device)

    recipes = torch.tensor(
        candidate_recipes,
        dtype=torch.long
    ).to(device)

    with torch.no_grad():
        predictions = model(users, recipes)

    # Top-K
    top_indices = torch.topk(
        predictions,
        k=top_k
    ).indices

    recommended_recipes = [
        candidate_recipes[i]
        for i in top_indices.cpu().numpy()
    ]

    return recommended_recipes

In [ ]:
# Uzimamo prvog korisnika koji postoji u test skupu
test_user = test_ratings["user_idx"].iloc[0]

recommendations = recommend_ncf(
    user_idx=test_user,
    model=model,
    num_recipes=num_recipes,
    user_seen_recipes=user_seen_recipes,
    top_k=10
)

print("User:", test_user)
print("Recommended recipe indexes:", recommendations)

In [ ]:
from sklearn.model_selection import train_test_split

# Odvajamo 10% trening podataka za validation
train_data, val_data = train_test_split(
    train_ratings,
    test_size=0.10,
    random_state=42
)

train_data = train_data.reset_index(drop=True)
val_data = val_data.reset_index(drop=True)

print("Train:", train_data.shape)
print("Validation:", val_data.shape)

In [ ]:
train_dataset = RecipeRatingDataset(train_data)
val_dataset = RecipeRatingDataset(val_data)

train_loader = DataLoader(
    train_dataset,
    batch_size=1024,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=1024,
    shuffle=False
)

print("Training samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))

In [ ]:
import torch.nn as nn


class NCF(nn.Module):

    def __init__(
            self,
            num_users,
            num_recipes,
            embedding_dim=32,
            dropout=0.2
    ):
        super().__init__()

        self.user_embedding = nn.Embedding(
            num_users,
            embedding_dim
        )

        self.recipe_embedding = nn.Embedding(
            num_recipes,
            embedding_dim
        )

        self.mlp = nn.Sequential(
            nn.Linear(embedding_dim * 2, 64),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(32, 16),
            nn.ReLU(),

            nn.Linear(16, 1)
        )

    def forward(self, user, recipe):

        user_vector = self.user_embedding(user)
        recipe_vector = self.recipe_embedding(recipe)

        x = torch.cat(
            [user_vector, recipe_vector],
            dim=1
        )

        output = self.mlp(x)

        return output.squeeze(1)

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = NCF(
    num_users=num_users,
    num_recipes=num_recipes,
    embedding_dim=32,
    dropout=0.2
).to(device)

criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001,
    weight_decay=1e-5
)

print("Device:", device)
print("Parameters:", sum(
    p.numel() for p in model.parameters()
))

In [ ]:
import time
import copy

num_epochs = 35
patience = 7

best_val_loss = float("inf")
best_model_state = None
epochs_without_improvement = 0

train_losses = []
val_losses = []

for epoch in range(num_epochs):

    start_time = time.time()

    # =========================
    # TRAIN
    # =========================

    model.train()

    total_train_loss = 0.0

    for users, recipes_batch, ratings_batch in train_loader:

        users = users.to(device)
        recipes_batch = recipes_batch.to(device)
        ratings_batch = ratings_batch.to(device)

        optimizer.zero_grad()

        predictions = model(
            users,
            recipes_batch
        )

        loss = criterion(
            predictions,
            ratings_batch
        )

        loss.backward()
        optimizer.step()

        total_train_loss += loss.item()

    avg_train_loss = total_train_loss / len(train_loader)

    # =========================
    # VALIDATION
    # =========================

    model.eval()

    total_val_loss = 0.0

    with torch.no_grad():

        for users, recipes_batch, ratings_batch in val_loader:

            users = users.to(device)
            recipes_batch = recipes_batch.to(device)
            ratings_batch = ratings_batch.to(device)

            predictions = model(
                users,
                recipes_batch
            )

            loss = criterion(
                predictions,
                ratings_batch
            )

            total_val_loss += loss.item()

    avg_val_loss = total_val_loss / len(val_loader)

    train_losses.append(avg_train_loss)
    val_losses.append(avg_val_loss)

    elapsed = time.time() - start_time

    print(
        f"Epoch {epoch + 1:02d}/{num_epochs} "
        f"- Train Loss: {avg_train_loss:.4f} "
        f"- Val Loss: {avg_val_loss:.4f} "
        f"- Time: {elapsed:.1f}s"
    )

    # =========================
    # BEST MODEL
    # =========================

    if avg_val_loss < best_val_loss:

        best_val_loss = avg_val_loss

        best_model_state = copy.deepcopy(
            model.state_dict()
        )

        epochs_without_improvement = 0

        print("  -> New best model!")

    else:

        epochs_without_improvement += 1

        print(
            f"  -> No improvement "
            f"({epochs_without_improvement}/{patience})"
        )

    # =========================
    # EARLY STOPPING
    # =========================

    if epochs_without_improvement >= patience:

        print("Early stopping triggered.")

        break

In [ ]:
class NCF(nn.Module):

    def __init__(
            self,
            num_users,
            num_recipes,
            embedding_dim=16,
            dropout=0.3
    ):
        super().__init__()

        self.user_embedding = nn.Embedding(
            num_users,
            embedding_dim
        )

        self.recipe_embedding = nn.Embedding(
            num_recipes,
            embedding_dim
        )

        self.mlp = nn.Sequential(
            nn.Linear(embedding_dim * 2, 32),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(16, 8),
            nn.ReLU(),

            nn.Linear(8, 1)
        )

    def forward(self, user, recipe):

        user_vector = self.user_embedding(user)
        recipe_vector = self.recipe_embedding(recipe)

        x = torch.cat(
            [user_vector, recipe_vector],
            dim=1
        )

        output = self.mlp(x)

        return output.squeeze(1)

In [ ]:
model = NCF(
    num_users=num_users,
    num_recipes=num_recipes,
    embedding_dim=16,
    dropout=0.3
).to(device)

criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.0005,
    weight_decay=1e-4
)

print("Device:", device)
print(
    "Parameters:",
    sum(p.numel() for p in model.parameters())
)

In [ ]:
import time
import copy

num_epochs = 35
patience = 7

best_val_loss = float("inf")
best_model_state = None
epochs_without_improvement = 0

train_losses = []
val_losses = []

for epoch in range(num_epochs):

    start_time = time.time()

    # =========================
    # TRAIN
    # =========================

    model.train()

    total_train_loss = 0.0

    for users, recipes_batch, ratings_batch in train_loader:

        users = users.to(device)
        recipes_batch = recipes_batch.to(device)
        ratings_batch = ratings_batch.to(device)

        optimizer.zero_grad()

        predictions = model(
            users,
            recipes_batch
        )

        loss = criterion(
            predictions,
            ratings_batch
        )

        loss.backward()
        optimizer.step()

        total_train_loss += loss.item()

    avg_train_loss = total_train_loss / len(train_loader)

    # =========================
    # VALIDATION
    # =========================

    model.eval()

    total_val_loss = 0.0

    with torch.no_grad():

        for users, recipes_batch, ratings_batch in val_loader:

            users = users.to(device)
            recipes_batch = recipes_batch.to(device)
            ratings_batch = ratings_batch.to(device)

            predictions = model(
                users,
                recipes_batch
            )

            loss = criterion(
                predictions,
                ratings_batch
            )

            total_val_loss += loss.item()

    avg_val_loss = total_val_loss / len(val_loader)

    train_losses.append(avg_train_loss)
    val_losses.append(avg_val_loss)

    elapsed = time.time() - start_time

    print(
        f"Epoch {epoch + 1:02d}/{num_epochs} "
        f"- Train Loss: {avg_train_loss:.4f} "
        f"- Val Loss: {avg_val_loss:.4f} "
        f"- Time: {elapsed:.1f}s"
    )

    # =========================
    # BEST MODEL
    # =========================

    if avg_val_loss < best_val_loss:

        best_val_loss = avg_val_loss

        best_model_state = copy.deepcopy(
            model.state_dict()
        )

        epochs_without_improvement = 0

        print("  -> New best model!")

    else:

        epochs_without_improvement += 1

        print(
            f"  -> No improvement "
            f"({epochs_without_improvement}/{patience})"
        )

    # =========================
    # EARLY STOPPING
    # =========================

    if epochs_without_improvement >= patience:

        print("Early stopping triggered.")

        break

In [ ]:
model.load_state_dict(best_model_state)
model.eval()

print(f"Best validation loss: {best_val_loss:.4f}")

In [ ]:
# Korisnici koji imaju istoriju u trening skupu
train_user_counts = train_data.groupby("user_idx").size()

# Korisnici koji imaju bar jedan validation rating
val_users = val_data["user_idx"].unique()

# Zadržavamo korisnike sa dovoljno istorije
eval_users = [
    user for user in val_users
    if train_user_counts.get(user, 0) >= 5
]

print("Validation users:", len(val_users))
print("Evaluation users:", len(eval_users))

In [ ]:
def recommend_ncf(
        user_idx,
        model,
        num_recipes,
        user_seen_recipes,
        top_k=10,
        recipe_batch_size=2048
):
    model.eval()

    seen = user_seen_recipes.get(user_idx, set())

    all_scores = []

    with torch.no_grad():

        for start in range(0, num_recipes, recipe_batch_size):

            end = min(
                start + recipe_batch_size,
                num_recipes
            )

            candidate_recipes = [
                r for r in range(start, end)
                if r not in seen
            ]

            if not candidate_recipes:
                continue

            users = torch.full(
                (len(candidate_recipes),),
                user_idx,
                dtype=torch.long,
                device=device
            )

            recipes = torch.tensor(
                candidate_recipes,
                dtype=torch.long,
                device=device
            )

            scores = model(users, recipes)

            all_scores.extend(
                zip(
                    candidate_recipes,
                    scores.cpu().numpy()
                )
            )

    # Sortiranje po predviđenoj oceni
    all_scores.sort(
        key=lambda x: x[1],
        reverse=True
    )

    return [
        recipe_idx
        for recipe_idx, score in all_scores[:top_k]
    ]

In [ ]:
test_user = eval_users[0]

recommendations = recommend_ncf(
    user_idx=test_user,
    model=model,
    num_recipes=num_recipes,
    user_seen_recipes=user_seen_recipes,
    top_k=10
)

print("User:", test_user)
print("Top-10:", recommendations)

In [ ]:
user_val = val_data[
    (val_data["user_idx"] == test_user) &
    (val_data["rating"] >= 4)
    ]

relevant_recipes = set(
    user_val["recipe_idx"]
)

print("Relevant recipes:", relevant_recipes)
print("Recommended:", set(recommendations))

In [ ]:
hits = len(
    set(recommendations) & relevant_recipes
)

precision_at_10 = hits / 10

recall_at_10 = (
    hits / len(relevant_recipes)
    if len(relevant_recipes) > 0
    else 0
)

print(f"Hits: {hits}")
print(f"Precision@10: {precision_at_10:.4f}")
print(f"Recall@10: {recall_at_10:.4f}")

In [ ]:
print("User:", test_user)

print("\nValidation ratings:")
print(
    val_data[
        val_data["user_idx"] == test_user
        ][["recipe_idx", "rating"]]
    .sort_values("rating", ascending=False)
    .head(20)
)

print("\nNCF recommendations:")
print(recommendations)

In [ ]:
# Prvih 20 preporuka sa njihovim score-ovima

model.eval()

seen = user_seen_recipes.get(test_user, set())

scores = []

with torch.no_grad():
    for recipe_idx in range(num_recipes):

        if recipe_idx in seen:
            continue

        user_tensor = torch.tensor(
            [test_user],
            dtype=torch.long,
            device=device
        )

        recipe_tensor = torch.tensor(
            [recipe_idx],
            dtype=torch.long,
            device=device
        )

        score = model(
            user_tensor,
            recipe_tensor
        ).item()

        scores.append(
            (recipe_idx, score)
        )

scores.sort(
    key=lambda x: x[1],
    reverse=True
)

print("Top 20 predictions:")
for recipe_idx, score in scores[:20]:
    print(
        f"Recipe {recipe_idx}: "
        f"{score:.4f}"
    )

In [ ]:
# Relevantni recepti korisnika iz validation skupa
relevant_recipes = set(
    val_data[
        (val_data["user_idx"] == test_user) &
        (val_data["rating"] >= 4)
        ]["recipe_idx"]
)

print("Relevant recipes:", relevant_recipes)

# Pozicija svakog relevantnog recepta u NCF rankingu
recipe_ranks = {
    recipe_idx: rank
    for rank, (recipe_idx, score) in enumerate(scores, start=1)
}

print("\nRank relevantnih recepata:")

for recipe in relevant_recipes:
    rank = recipe_ranks.get(recipe)

    if rank is not None:
        print(
            f"Recipe {recipe}: "
            f"Rank = {rank}"
        )

In [ ]:
for recipe in relevant_recipes:
    print(
        f"Recipe {recipe}: "
        f"seen = {recipe in user_seen_recipes.get(test_user, set())}"
    )

In [ ]:
user_seen_recipes = (
    train_data
    .groupby("user_idx")["recipe_idx"]
    .apply(set)
    .to_dict()
)

print("user_seen_recipes recreated from TRAIN only.")

In [ ]:
for recipe in relevant_recipes:
    print(
        f"Recipe {recipe}: "
        f"seen = {recipe in user_seen_recipes.get(test_user, set())}"
    )

In [ ]:
recommendations = recommend_ncf(
    user_idx=test_user,
    model=model,
    num_recipes=num_recipes,
    user_seen_recipes=user_seen_recipes,
    top_k=10
)

print("Recommendations:")
print(recommendations)

In [ ]:
hits = len(
    set(recommendations) & relevant_recipes
)

precision_at_10 = hits / 10

recall_at_10 = (
    hits / len(relevant_recipes)
    if len(relevant_recipes) > 0
    else 0
)

print(f"Hits: {hits}")
print(f"Precision@10: {precision_at_10:.4f}")
print(f"Recall@10: {recall_at_10:.4f}")

In [ ]:
import numpy as np
import time

model.eval()

precisions = []
recalls = []
hit_counts = []

start_time = time.time()

for i, user_idx in enumerate(eval_users):

    # Relevantni recepti iz validation skupa
    relevant = set(
        val_data[
            (val_data["user_idx"] == user_idx) &
            (val_data["rating"] >= 4)
            ]["recipe_idx"]
    )

    # Ako korisnik nema relevantan validation recept,
    # preskačemo ga
    if len(relevant) == 0:
        continue

    # Top-10 preporuke
    recommended = recommend_ncf(
        user_idx=user_idx,
        model=model,
        num_recipes=num_recipes,
        user_seen_recipes=user_seen_recipes,
        top_k=10
    )

    recommended = set(recommended)

    hits = len(recommended & relevant)

    precision = hits / 10
    recall = hits / len(relevant)

    precisions.append(precision)
    recalls.append(recall)
    hit_counts.append(hits)

    if (i + 1) % 100 == 0:
        print(
            f"Processed {i + 1}/{len(eval_users)} users"
        )

elapsed = time.time() - start_time

print("\n==============================")
print("NCF EVALUATION")
print("==============================")

print(f"Users evaluated: {len(precisions)}")
print(f"Precision@10: {np.mean(precisions):.4f}")
print(f"Recall@10:    {np.mean(recalls):.4f}")
print(f"Average Hits: {np.mean(hit_counts):.4f}")
print(f"Time: {elapsed:.1f}s")

In [ ]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader


class BPRDataset(Dataset):

    def __init__(
            self,
            ratings_df,
            num_recipes
    ):
        self.num_recipes = num_recipes

        # Pozitivni recepti: rating >= 4
        positive_df = ratings_df[
            ratings_df["rating"] >= 4
            ].copy()

        self.positive_pairs = list(
            zip(
                positive_df["user_idx"].astype(int),
                positive_df["recipe_idx"].astype(int)
            )
        )

        # Recepti koje je korisnik već ocenio
        self.user_seen = (
            ratings_df
            .groupby("user_idx")["recipe_idx"]
            .apply(set)
            .to_dict()
        )

    def __len__(self):
        return len(self.positive_pairs)

    def __getitem__(self, idx):

        user, positive_recipe = self.positive_pairs[idx]

        seen = self.user_seen.get(user, set())

        # Nasumičan negativan recept
        negative_recipe = np.random.randint(
            0,
            self.num_recipes
        )

        while negative_recipe in seen:
            negative_recipe = np.random.randint(
                0,
                self.num_recipes
            )

        return (
            torch.tensor(user, dtype=torch.long),
            torch.tensor(
                positive_recipe,
                dtype=torch.long
            ),
            torch.tensor(
                negative_recipe,
                dtype=torch.long
            )
        )

In [ ]:
bpr_dataset = BPRDataset(
    train_data,
    num_recipes
)

bpr_loader = DataLoader(
    bpr_dataset,
    batch_size=1024,
    shuffle=True
)

print("BPR samples:", len(bpr_dataset))

In [ ]:
class NCF(nn.Module):

    def __init__(
            self,
            num_users,
            num_recipes,
            embedding_dim=32,
            dropout=0.2
    ):
        super().__init__()

        self.user_embedding = nn.Embedding(
            num_users,
            embedding_dim
        )

        self.recipe_embedding = nn.Embedding(
            num_recipes,
            embedding_dim
        )

        self.mlp = nn.Sequential(
            nn.Linear(embedding_dim * 2, 64),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(32, 16),
            nn.ReLU(),

            nn.Linear(16, 1)
        )

    def forward(self, user, recipe):

        user_vector = self.user_embedding(user)
        recipe_vector = self.recipe_embedding(recipe)

        x = torch.cat(
            [user_vector, recipe_vector],
            dim=1
        )

        return self.mlp(x).squeeze(1)

In [ ]:
def bpr_loss(
        positive_scores,
        negative_scores
):
    return -torch.mean(
        torch.log(
            torch.sigmoid(
                positive_scores - negative_scores
            ) + 1e-8
        )
    )

In [ ]:
model = NCF(
    num_users=num_users,
    num_recipes=num_recipes,
    embedding_dim=32,
    dropout=0.2
).to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001,
    weight_decay=1e-5
)

print("Device:", device)
print(
    "Parameters:",
    sum(p.numel() for p in model.parameters())
)

In [ ]:
num_epochs = 35

for epoch in range(num_epochs):

    model.train()

    total_loss = 0.0

    for users, positive_recipes, negative_recipes in bpr_loader:

        users = users.to(device)
        positive_recipes = positive_recipes.to(device)
        negative_recipes = negative_recipes.to(device)

        optimizer.zero_grad()

        positive_scores = model(
            users,
            positive_recipes
        )

        negative_scores = model(
            users,
            negative_recipes
        )

        loss = bpr_loss(
            positive_scores,
            negative_scores
        )

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(bpr_loader)

    print(
        f"Epoch {epoch + 1:02d}/{num_epochs} "
        f"- BPR Loss: {avg_loss:.4f}"
    )

In [ ]:
model.eval()

with torch.no_grad():
    recipe_embeddings = model.recipe_embedding.weight

print(recipe_embeddings.shape)

In [ ]:
def fast_recommend_ncf(
        user_idx,
        model,
        num_recipes,
        user_seen_recipes,
        top_k=10
):
    model.eval()

    seen = user_seen_recipes.get(
        user_idx,
        set()
    )

    with torch.no_grad():

        user_tensor = torch.tensor(
            [user_idx],
            dtype=torch.long,
            device=device
        )

        user_embedding = model.user_embedding(
            user_tensor
        )

        user_embedding = user_embedding.expand(
            num_recipes,
            -1
        )

        recipe_indices = torch.arange(
            num_recipes,
            device=device
        )

        recipe_embedding = model.recipe_embedding(
            recipe_indices
        )

        x = torch.cat(
            [
                user_embedding,
                recipe_embedding
            ],
            dim=1
        )

        scores = model.mlp(x).squeeze(1)

        # Izbacujemo recepte koje je korisnik već
        # imao u TRAIN skupu
        if seen:
            seen_tensor = torch.tensor(
                list(seen),
                dtype=torch.long,
                device=device
            )

            scores[seen_tensor] = -float("inf")

        top_scores, top_indices = torch.topk(
            scores,
            k=top_k
        )

    return top_indices.cpu().numpy()

In [ ]:
test_user = eval_users[0]

recommendations = fast_recommend_ncf(
    user_idx=test_user,
    model=model,
    num_recipes=num_recipes,
    user_seen_recipes=user_seen_recipes,
    top_k=10
)

print("User:", test_user)
print("Top-10:", recommendations)

In [ ]:
relevant_recipes = set(
    val_data[
        (val_data["user_idx"] == test_user) &
        (val_data["rating"] >= 4)
        ]["recipe_idx"]
)

hits = len(
    set(recommendations) & relevant_recipes
)

precision_at_10 = hits / 10

recall_at_10 = (
    hits / len(relevant_recipes)
    if relevant_recipes
    else 0
)

print("Relevant:", relevant_recipes)
print("Recommended:", recommendations)
print("Hits:", hits)
print(f"Precision@10: {precision_at_10:.4f}")
print(f"Recall@10: {recall_at_10:.4f}")

In [ ]:
print("USER:", test_user)

print("\nTRAIN positives:")
print(
    train_data[
        (train_data["user_idx"] == test_user) &
        (train_data["rating"] >= 4)
        ][["recipe_idx", "rating"]]
    .sort_values("rating", ascending=False)
    .head(20)
)

print("\nVALIDATION positives:")
print(
    val_data[
        (val_data["user_idx"] == test_user) &
        (val_data["rating"] >= 4)
        ][["recipe_idx", "rating"]]
    .sort_values("rating", ascending=False)
)

print("\nNCF recommendations:")
print(recommendations)

In [ ]:
print("Users:", num_users)
print("Recipes:", num_recipes)
print("Train ratings:", len(train_data))
print("Validation ratings:", len(val_data))

print(
    "Average train ratings/user:",
    len(train_data) / num_users
)

print(
    "Average train ratings/recipe:",
    len(train_data) / num_recipes
)

In [ ]:
for user in eval_users[:5]:

    recs = fast_recommend_ncf(
        user_idx=user,
        model=model,
        num_recipes=num_recipes,
        user_seen_recipes=user_seen_recipes,
        top_k=10
    )

    print(f"User {user}: {recs}")

In [ ]:
from sklearn.decomposition import TruncatedSVD

# tfidf_matrix već imamo iz Content-Based modela

print("Original TF-IDF shape:", tfidf_matrix.shape)

svd = TruncatedSVD(
    n_components=64,
    random_state=42
)

recipe_content_features = svd.fit_transform(tfidf_matrix)

print("Reduced content shape:", recipe_content_features.shape)

In [ ]:
[name for name in dir() if not name.startswith("_")]

In [ ]:
print("recipe type:", type(recipe))

if hasattr(recipe, "shape"):
    print("recipe shape:", recipe.shape)

if hasattr(recipe, "columns"):
    print("recipe columns:", recipe.columns.tolist())

In [ ]:
print("ratings type:", type(ratings))

if hasattr(ratings, "shape"):
    print("ratings shape:", ratings.shape)

if hasattr(ratings, "columns"):
    print("ratings columns:", ratings.columns.tolist())

In [ ]:
from pathlib import Path

project_path = Path(r"C:\Users\Nikola\Desktop\FoodRecommendationSystem")

for file in project_path.rglob("*"):
    if file.is_file() and file.suffix.lower() in [".csv", ".json", ".parquet"]:
        print(file)

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import copy
import time


class NCFv2(nn.Module):

    def __init__(
            self,
            num_users,
            num_recipes,
            content_embeddings,
            embedding_dim=32
    ):
        super().__init__()

        self.user_embedding = nn.Embedding(
            num_users,
            embedding_dim
        )

        self.recipe_embedding = nn.Embedding(
            num_recipes,
            embedding_dim
        )

        # Content embedding koji smo već napravili
        self.register_buffer(
            "content_embeddings",
            torch.tensor(
                content_embeddings,
                dtype=torch.float32
            )
        )

        content_dim = content_embeddings.shape[1]

        # User + recipe + content
        input_dim = embedding_dim * 2 + content_dim

        self.mlp = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(64, 32),
            nn.ReLU(),

            nn.Linear(32, 1)
        )

        self._init_weights()

    def _init_weights(self):
        nn.init.normal_(self.user_embedding.weight, std=0.01)
        nn.init.normal_(self.recipe_embedding.weight, std=0.01)

    def forward(self, user_ids, recipe_ids):

        user_vec = self.user_embedding(user_ids)

        recipe_vec = self.recipe_embedding(recipe_ids)

        content_vec = self.content_embeddings[recipe_ids]

        x = torch.cat(
            [
                user_vec,
                recipe_vec,
                content_vec
            ],
            dim=1
        )

        return self.mlp(x).squeeze(1)

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)

model_v2 = NCFv2(
    num_users=num_users,
    num_recipes=num_recipes,
    content_embeddings=recipe_embeddings,
    embedding_dim=32
).to(device)

print(
    "V2 parameters:",
    sum(p.numel() for p in model_v2.parameters())
)

In [ ]:
class BPRDatasetV2(torch.utils.data.Dataset):

    def __init__(
            self,
            ratings,
            num_recipes
    ):
        self.num_recipes = num_recipes

        positive = ratings[
            ratings["rating"] >= 4
            ][["user_idx", "recipe_idx"]].drop_duplicates()

        self.users = positive["user_idx"].values
        self.positive_items = positive["recipe_idx"].values

        # Sve što je korisnik već ocenio
        self.user_seen = {}

        for user, recipe in zip(
                ratings["user_idx"],
                ratings["recipe_idx"]
        ):
            if user not in self.user_seen:
                self.user_seen[user] = set()

            self.user_seen[user].add(recipe)

    def __len__(self):
        return len(self.users)

    def __getitem__(self, idx):

        user = int(self.users[idx])
        positive = int(self.positive_items[idx])

        # Random negative
        negative = np.random.randint(
            0,
            self.num_recipes
        )

        while negative in self.user_seen[user]:
            negative = np.random.randint(
                0,
                self.num_recipes
            )

        return (
            user,
            positive,
            negative
        )

In [ ]:
bpr_dataset_v2 = BPRDatasetV2(
    train_ratings,
    num_recipes
)

bpr_loader_v2 = torch.utils.data.DataLoader(
    bpr_dataset_v2,
    batch_size=1024,
    shuffle=True,
    num_workers=0
)

print("Training samples:", len(bpr_dataset_v2))

In [ ]:
optimizer_v2 = torch.optim.Adam(
    model_v2.parameters(),
    lr=0.001,
    weight_decay=1e-5
)

epochs = 35
patience = 7

best_loss = float("inf")
best_state = None
no_improvement = 0


for epoch in range(epochs):

    model_v2.train()

    start_time = time.time()

    total_loss = 0
    batches = 0

    for users, positives, negatives in bpr_loader_v2:

        users = users.to(device)
        positives = positives.to(device)
        negatives = negatives.to(device)

        optimizer_v2.zero_grad()

        positive_scores = model_v2(
            users,
            positives
        )

        negative_scores = model_v2(
            users,
            negatives
        )

        # BPR loss
        loss = -torch.mean(
            torch.log(
                torch.sigmoid(
                    positive_scores - negative_scores
                ) + 1e-8
            )
        )

        loss.backward()

        optimizer_v2.step()

        total_loss += loss.item()
        batches += 1

    avg_loss = total_loss / batches

    elapsed = time.time() - start_time

    print(
        f"Epoch {epoch+1:02d}/{epochs} "
        f"- BPR Loss: {avg_loss:.4f} "
        f"- Time: {elapsed:.1f}s"
    )

    # Early stopping na training loss
    if avg_loss < best_loss:
        best_loss = avg_loss
        best_state = copy.deepcopy(
            model_v2.state_dict()
        )
        no_improvement = 0

        print("  -> New best model!")

    else:
        no_improvement += 1

        print(
            f"  -> No improvement "
            f"({no_improvement}/{patience})"
        )

        if no_improvement >= patience:
            print("Early stopping triggered.")
            break


# Vrati najbolji model
model_v2.load_state_dict(best_state)

print("\nBest BPR loss:", best_loss)

In [ ]:
print(type(val_users))
print(type(val_data))

if hasattr(val_users, "shape"):
    print("val_users shape:", val_users.shape)

if hasattr(val_data, "shape"):
    print("val_data shape:", val_data.shape)

print("val_users first:", val_users[:5])

In [ ]:
# ==========================================
# BPR V2 - EVALUATION
# ==========================================

model.eval()

precision_scores = []
recall_scores = []
hit_counts = []

for user in eval_users:

    # User -> internal index
    if user not in user_to_idx:
        continue

    user_idx = user_to_idx[user]

    # Recepti koje je korisnik već video u treningu
    seen = user_seen_recipes.get(user_idx, set())

    # Relevantni recepti iz validation skupa
    user_val = val_data[
        val_data["user_id"] == user
        ]

    # Samo pozitivне оцене (4 или 5)
    relevant = set(
        user_val[
            user_val["rating"] >= 4
            ]["recipe_idx"].tolist()
    )

    if len(relevant) == 0:
        continue

    # Kandidati: svi recepti koje korisnik nije video
    candidates = [
        r for r in range(num_recipes)
        if r not in seen
    ]

    user_tensor = torch.tensor(
        [user_idx] * len(candidates),
        dtype=torch.long,
        device=device
    )

    recipe_tensor = torch.tensor(
        candidates,
        dtype=torch.long,
        device=device
    )

    with torch.no_grad():
        scores = model(
            user_tensor,
            recipe_tensor
        ).cpu().numpy()

    # Top 10
    top_indices = np.argsort(scores)[-10:][::-1]

    recommended = [
        candidates[i]
        for i in top_indices
    ]

    # Hits
    hits = len(
        set(recommended) & relevant
    )

    precision_scores.append(
        hits / 10
    )

    recall_scores.append(
        hits / len(relevant)
    )

    hit_counts.append(hits)


print("\n==============================")
print("BPR V2 EVALUATION")
print("==============================")

print(
    f"Users evaluated: {len(precision_scores)}"
)

print(
    f"Precision@10: "
    f"{np.mean(precision_scores):.4f}"
)

print(
    f"Recall@10:    "
    f"{np.mean(recall_scores):.4f}"
)

print(
    f"Average Hits: "
    f"{np.mean(hit_counts):.4f}"
)

In [ ]:
# ==========================================
# BPR V2 - ISPRAVNA EVALUACIJA
# Isto pravilo za sve korisnike
# ==========================================

model_v2.eval()

precision_scores = []
recall_scores = []
hit_counts = []

for user in val_users:

    # originalni user ID -> internal index
    if user not in user_to_idx:
        continue

    user_idx = user_to_idx[user]

    # Validation pozitivni recepti
    user_val = val_data[
        (val_data["user_id"] == user) &
        (val_data["rating"] >= 4)
        ]

    relevant = set(
        user_val["recipe_idx"].astype(int).tolist()
    )

    if len(relevant) == 0:
        continue

    # Recepti koje je korisnik već imao u TRAIN skupu
    train_seen = train_ratings[
        train_ratings["user_id"] == user
        ]["recipe_idx"].astype(int).tolist()

    seen = set(train_seen)

    # Kandidati
    candidates = np.array([
        r for r in range(num_recipes)
        if r not in seen
    ], dtype=np.int64)

    # Score kandidata u batch-evima
    scores = []

    with torch.no_grad():

        for start in range(0, len(candidates), 4096):

            batch_recipes = candidates[
                            start:start + 4096
                            ]

            user_tensor = torch.full(
                (len(batch_recipes),),
                user_idx,
                dtype=torch.long,
                device=device
            )

            recipe_tensor = torch.tensor(
                batch_recipes,
                dtype=torch.long,
                device=device
            )

            batch_scores = model_v2(
                user_tensor,
                recipe_tensor
            ).cpu().numpy()

            scores.append(batch_scores)

    scores = np.concatenate(scores)

    # Top 10
    top10_idx = np.argpartition(
        scores,
        -10
    )[-10:]

    top10_idx = top10_idx[
        np.argsort(scores[top10_idx])[::-1]
    ]

    recommended = candidates[top10_idx]

    # Hits
    hits = len(
        set(recommended) & relevant
    )

    precision_scores.append(
        hits / 10.0
    )

    recall_scores.append(
        hits / len(relevant)
    )

    hit_counts.append(hits)


print("\n==============================")
print("BPR V2 EVALUATION")
print("==============================")

print(
    f"Users evaluated: "
    f"{len(precision_scores)}"
)

print(
    f"Precision@10: "
    f"{np.mean(precision_scores):.6f}"
)

print(
    f"Recall@10: "
    f"{np.mean(recall_scores):.6f}"
)

print(
    f"Average Hits: "
    f"{np.mean(hit_counts):.6f}"
)

print(
    f"Users with at least 1 hit: "
    f"{sum(h > 0 for h in hit_counts)}"
)

In [ ]:
print("num_users:", num_users)
print("num_recipes:", num_recipes)

print("\nMappings:")
print("user_to_idx:", len(user_to_idx))
print("recipe_to_idx:", len(recipe_to_idx))

print("\nTRAIN columns:")
print(train_ratings.columns.tolist())

print("\nVAL columns:")
print(val_data.columns.tolist())

print("\nExample train:")
print(train_ratings.head())

print("\nExample validation:")
print(val_data.head())

In [ ]:
print("\nUSER 11035 CHECK")

print("Original user ID:", 11035)
print("Mapped user:", user_to_idx.get(11035))

print("\nValidation recipes:")
print(
    val_data[
        (val_data["user_id"] == 11035) &
        (val_data["rating"] >= 4)
        ][["user_id", "recipe_idx", "rating"]]
)

In [ ]:
user = 11035
recipe = 8096

user_idx = user_to_idx[user]

model_v2.eval()

with torch.no_grad():

    score = model_v2(
        torch.tensor(
            [user_idx],
            dtype=torch.long,
            device=device
        ),
        torch.tensor(
            [recipe],
            dtype=torch.long,
            device=device
        )
    ).item()

print("User:", user)
print("User idx:", user_idx)
print("Recipe:", recipe)
print("Model score:", score)